In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
transactions = pd.read_excel("/Workspace/Users/musindo.alice@gmail.com/FreshMart-Store-Analysis/1.Project Description/1788896704791_FreshMart_Dataset_1.xlsx", sheet_name="Transactions")
display(transactions)

In [0]:
# Inspecting the data
transactions.shape
transactions.isnull().sum()
transactions["customer_id"].isnull().sum() / len(transactions)

In [0]:
print("Missing dates:")
print(transactions["transaction_date"].isna().sum())

In [0]:
transactions.groupby("month").size().reset_index(
    name="transaction_count"
)

In [0]:
transactions["year"] = transactions["transaction_date"].dt.year
transactions["month_number"] = transactions["transaction_date"].dt.month
transactions["month_name"] = transactions["transaction_date"].dt.month_name()

transactions.groupby(
    ["year", "month_number", "month_name"]
).size().reset_index(
    name="transaction_count"
)

In [0]:
# Create half_year column from transaction_date
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["half_year"] = transactions["transaction_date"].dt.year.astype(str) + "-H" + ((transactions["transaction_date"].dt.month - 1) // 6 + 1).astype(str)

kpi = transactions.groupby("half_year").agg(
    transactions=("transaction_id","count"),
    revenue=("basket_value_zar","sum"),
    avg_basket=("basket_value_zar","mean"),
    avg_items=("num_items","mean")
)
kpi["value_per_item"] = kpi["avg_basket"] / kpi["avg_items"]

In [0]:
import numpy as np

transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["half_year"] = np.where(
    transactions["transaction_date"].dt.month <= 6,
    "H1",
    "H2"
)


In [0]:
#Checking KPI
kpi = transactions.groupby("half_year").agg(
    transactions=("transaction_id", "count"),
    revenue=("basket_value_zar", "sum"),
    avg_basket=("basket_value_zar", "mean"),
    avg_items=("num_items", "mean")
)

kpi["value_per_item"] = (
    kpi["avg_basket"] /
    kpi["avg_items"]
)

kpi

In [0]:
# Analysis by province
transactions["half"]=transactions["transaction_date"].dt.month.apply(lambda x: "H1" if x <= 6 else "H2")
transactions.groupby("half").agg(
    vol=("transaction_id","count"),
    avg_basket=("basket_value_zar", "mean"),
    avg_items=("num_items", "mean"),
    total_revenue=("basket_value_zar", "sum")
)
 

In [0]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["half"]=transactions["transaction_date"].dt.strftime("%Y-%m").str[:7]
# Analysis by province
stores = spark.table("workspace.default.freshmart_stores").toPandas()
df = transactions.merge(stores, on="store_id", how="left")
df.groupby("province").agg(
    transactions=("transaction_id", "count"),
    revenue=("basket_value_zar", "sum"),
    avg_basket=("basket_value_zar", "mean"),
    avg_items=("num_items", "mean")
).sort_values("avg_basket", ascending=False)

In [0]:
# Analysis by loyalty member status
transactions.groupby("is_loyalty_member").agg(
    transactions=("transaction_id", "count"),
    revenue=("basket_value_zar", "sum"),
    avg_basket=("basket_value_zar", "mean"),
    avg_items=("num_items", "mean")
)


In [0]:
# Analysis by loyalty member spend
transactions.groupby(["is_loyalty_member","half"]).agg(
   volume=("transaction_id", "count"),
   avg_basket=("basket_value_zar", "mean"),
)

In [0]:
# Convert transaction_date to proper datetime
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"],
    errors="coerce"
)

# Check the date range
print("Minimum date:", transactions["transaction_date"].min())
print("Maximum date:", transactions["transaction_date"].max())

# Create detailed date columns
transactions["year"] = transactions["transaction_date"].dt.year

transactions["month_number"] = transactions["transaction_date"].dt.month

transactions["month_name"] = transactions["transaction_date"].dt.month_name()

transactions["year_month"] = (
    transactions["transaction_date"].dt.to_period("M")
)

transactions["quarter"] = (
    transactions["transaction_date"].dt.quarter
)

transactions["day"] = transactions["transaction_date"].dt.day

transactions["day_name"] = (
    transactions["transaction_date"].dt.day_name()
)

transactions["day_of_week"] = (
    transactions["transaction_date"].dt.dayofweek
)

transactions["week_number"] = (
    transactions["transaction_date"].dt.isocalendar().week
)

In [0]:
transactions[
    [
        "transaction_date",
        "year",
        "month_number",
        "month_name",
        "year_month",
        "quarter",
        "day",
        "day_name",
        "day_of_week",
        "week_number"
    ]
].head(20)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

transactions = spark.table("workspace.default.freshmart_transactions").toPandas()
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["month"] = transactions["transaction_date"].dt.to_period("M")

monthly = (
    transactions
    .groupby("month")
    .agg(
        transactions=("transaction_id", "count"),
        avg_basket=("basket_value_zar", "mean"),
        avg_items=("num_items", "mean")
    )
    .reset_index()
)

# Convert Period to string for plotting
monthly["month"] = monthly["month"].astype(str)

monthly

#visual 1 Transaction volume increased through 2025
plt.figure(figsize=(12,6))

plt.plot(
    monthly["month"],
    monthly["transactions"]
)

plt.title("Monthly Transaction Volume")
plt.xlabel("Month")
plt.ylabel("Transactions")
plt.xticks(rotation=45)
plt.show()

#visual 2 Average basket value fell steadily across the year
plt.figure(figsize=(12,6))

plt.plot(
    monthly["month"],
    monthly["avg_basket"]
)

plt.title("Average Basket Value Trend")
plt.xlabel("Month")
plt.ylabel("Average Basket Value (ZAR)")
plt.xticks(rotation=45)
plt.show()

#visual 3 Customers are buying fewer items per transaction
plt.figure(figsize=(12,6))

plt.plot(
    monthly["month"],
    monthly["avg_items"]
)

plt.title("Average Items per Basket")
plt.xlabel("Month")
plt.ylabel("Average Items")
plt.xticks(rotation=45)
plt.show()

In [0]:
h1_h2 = (
    transactions
    .groupby("half_year")
    .agg(
        transactions=("transaction_id", "count"),
        avg_basket=("basket_value_zar", "mean"),
        avg_items=("num_items", "mean"),
        revenue=("basket_value_zar", "sum")
    )
    .reset_index()
)

h1_h2

#visualise 2
plt.figure(figsize=(8,5))

plt.bar(
    h1_h2["half_year"],
    h1_h2["avg_basket"]
)

plt.title("H1 vs H2 Average Basket Value")
plt.xlabel("Period")
plt.ylabel("Average Basket Value (ZAR)")

plt.show()

In [0]:
loyalty_analysis = (
    transactions
    .groupby(["half_year", "is_loyalty_member"])
    .agg(
        avg_basket=("basket_value_zar", "mean"),
        avg_items=("num_items", "mean"),
        transactions=("transaction_id", "count")
    )
    .reset_index()
)

loyalty_analysis
#visual
plt.figure(figsize=(10,6))

for group in loyalty_analysis["is_loyalty_member"].unique():

    data = loyalty_analysis[
        loyalty_analysis["is_loyalty_member"] == group
    ]

    plt.plot(
        data["half_year"],
        data["avg_basket"],
        marker="o",
        label=group
    )

plt.title("Average Basket Value: Loyalty vs Non-Loyalty")
plt.xlabel("Period")
plt.ylabel("Average Basket Value (ZAR)")
plt.legend()

plt.show()

In [0]:
monthly_transactions = (
    transactions
    .groupby("year_month")
    .size()
    .reset_index(name="transaction_count")
)

monthly_transactions

# visualisation
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(
    monthly_transactions["year_month"].astype(str),
    monthly_transactions["transaction_count"]
)

plt.title("Monthly Transaction Trend")
plt.xlabel("Month")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=45)
plt.show()

In [0]:
transactions["day_name"] = (
    transactions["transaction_date"].dt.day_name()
)


day_transactions = (
    transactions
    .groupby("day_name")
    .size()
    .reset_index(name="transaction_count")
)

day_transactions

#visualise
plt.figure(figsize=(10, 6))

plt.bar(
    day_transactions["day_name"],
    day_transactions["transaction_count"]
)

plt.title("Transactions by Day of Week")
plt.xlabel("Day")
plt.ylabel("Transactions")

plt.show()

In [0]:
monthly_summary = (
    transactions
    .groupby(["month_number", "month_name"])
    .size()
    .reset_index(name="transactions")
    .sort_values("month_number")
)

monthly_summary

#visualise
plt.figure(figsize=(12, 6))

plt.bar(
    monthly_summary["month_name"],
    monthly_summary["transactions"]
)

plt.title("Transactions by Month")
plt.xlabel("Month")
plt.ylabel("Transactions")
plt.xticks(rotation=45)

plt.show()

In [0]:
stores = spark.table("workspace.default.freshmart_stores").toPandas()
transactions = transactions.merge(stores[["store_id", "has_nearby_competitor"]], on="store_id", how="left")

competitor_analysis = (
    transactions
    .groupby(["half_year", "has_nearby_competitor"])
    .agg(
        avg_basket=("basket_value_zar", "mean"),
        avg_items=("num_items", "mean"),
        transactions=("transaction_id", "count")
    )
    .reset_index()
)

competitor_analysis

In [0]:
stores = spark.table("workspace.default.freshmart_stores").toPandas()
transactions = transactions.merge(stores[["store_id", "province"]], on="store_id", how="left")

province_analysis = (
    transactions
    .groupby(["half_year", "province"])
    .agg(
        avg_basket=("basket_value_zar", "mean"),
        avg_items=("num_items", "mean")
    )
    .reset_index()
)

province_analysis